# 02 — Groupement Temporel des Scènes

Regroupe les photos en **moments de vie** basés sur les timestamps EXIF :

- Un écart > **6h** entre deux photos consécutives = nouveau groupe
- Les photos **sans date EXIF** sont rattachées au groupe visuellement le plus proche (cosine similarity)
- Chaque groupe est nommé par son créneau horaire : `Soirée · 15 août 2023`

**Input** : `data/warehouse/memory_album/photo_embeddings/`  
**Output** : `data/warehouse/memory_album/scenes/` + `scene_centroids/` (Delta)

In [1]:
# ── 0. SETUP ──────────────────────────────────────────────────────────────────
import sys, os

_d = os.path.abspath('')
while not os.path.exists(os.path.join(_d, 'config.py')):
    _p = os.path.dirname(_d)
    if _p == _d: raise RuntimeError("config.py introuvable")
    _d = _p
sys.path.insert(0, _d)

import yaml
import numpy as np
import pandas as pd
from pyspark.sql import functions as F

from config import build_spark_session, MEMORY_ALBUM_DIR

EMBEDDINGS_DIR = os.path.join(MEMORY_ALBUM_DIR, 'photo_embeddings')
SCENES_DIR     = os.path.join(MEMORY_ALBUM_DIR, 'scenes')
CENTROIDS_DIR  = os.path.join(MEMORY_ALBUM_DIR, 'scene_centroids')

print(f"Input  : {EMBEDDINGS_DIR}")
print(f"Output : {SCENES_DIR}")
print(f"        {CENTROIDS_DIR}")

Input  : /opt/spark/data/warehouse/memory_album/photo_embeddings
Output : /opt/spark/data/warehouse/memory_album/scenes
        /opt/spark/data/warehouse/memory_album/scene_centroids


In [2]:
# ── 1. PARAMÈTRES ─────────────────────────────────────────────────────────────
_cfg_path = os.path.join(_d, 'config.yaml')
with open(_cfg_path, encoding='utf-8') as _f:
    _cfg = yaml.safe_load(_f)

_ma = _cfg.get('memory_album', {})

# Seuil de séparation entre deux moments (en heures)
GAP_HOURS = float(_ma.get('scene_gap_hours', 6.0))
GAP_SEC   = GAP_HOURS * 3600

print(f"Seuil gap temporel : {GAP_HOURS}h")

Seuil gap temporel : 6.0h


In [3]:
# ── 2. SESSION SPARK ──────────────────────────────────────────────────────────
spark = build_spark_session('MyDigitalTwin-MemoryAlbum-Clustering')
spark.sparkContext.setLogLevel('WARN')
print(f"Spark {spark.version}")

Spark 3.5.5


26/05/12 16:34:13 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [4]:
# ── 3. CHARGEMENT DES EMBEDDINGS ──────────────────────────────────────────────
from delta.tables import DeltaTable

if DeltaTable.isDeltaTable(spark, EMBEDDINGS_DIR):
    df = spark.read.format('delta').load(EMBEDDINGS_DIR)
else:
    df = spark.read.parquet(EMBEDDINGS_DIR)

total = df.count()
print(f"{total} photos chargées")
df.printSchema()

rows = df.select(
    'photo_id', 'path', 'filename', 'exif_date',
    'lat', 'lon', 'caption', 'embedding', 'has_gps'
).toPandas()

rows['exif_date'] = pd.to_datetime(rows['exif_date'], errors='coerce')
rows['caption']   = rows['caption'].fillna('')

n_dated   = rows['exif_date'].notna().sum()
n_undated = rows['exif_date'].isna().sum()
print(f"Avec date EXIF : {n_dated} | Sans date : {n_undated}")

26/05/12 16:34:26 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


64 photos chargées
root
 |-- photo_id: string (nullable = true)
 |-- path: string (nullable = true)
 |-- filename: string (nullable = true)
 |-- exif_date: timestamp (nullable = true)
 |-- lat: float (nullable = true)
 |-- lon: float (nullable = true)
 |-- caption: string (nullable = true)
 |-- embedding: array (nullable = true)
 |    |-- element: float (containsNull = true)
 |-- has_gps: boolean (nullable = true)
 |-- model_used: string (nullable = true)

Avec date EXIF : 49 | Sans date : 15


In [5]:
# ── 4. GROUPEMENT TEMPOREL (gap > GAP_HOURS = nouveau groupe) ─────────────────

MONTH_FR = ['jan', 'fév', 'mar', 'avr', 'mai', 'juin',
            'juil', 'août', 'sep', 'oct', 'nov', 'déc']

def time_slot(dt) -> str:
    h = dt.hour
    if   6  <= h < 12: return 'Matin'
    elif 12 <= h < 18: return 'Après-midi'
    elif 18 <= h < 23: return 'Soirée'
    else:              return 'Nuit'

def group_label(dt) -> str:
    return f"{time_slot(dt)} · {dt.day} {MONTH_FR[dt.month-1]} {dt.year}"

dated   = rows[rows['exif_date'].notna()].sort_values('exif_date').copy()
undated = rows[rows['exif_date'].isna()].copy()

groups = []
current = []

for idx, row in dated.iterrows():
    if not current:
        current.append(idx)
    else:
        prev_dt = rows.loc[current[-1], 'exif_date']
        gap = (row['exif_date'] - prev_dt).total_seconds()
        if gap > GAP_SEC:
            groups.append(current)
            current = [idx]
        else:
            current.append(idx)

if current:
    groups.append(current)

rows['scene_id']   = -1
rows['scene_name'] = ''

for gid, grp_indices in enumerate(groups):
    first_dt = rows.loc[grp_indices[0], 'exif_date']
    name     = group_label(first_dt)
    rows.loc[grp_indices, 'scene_id']   = gid
    rows.loc[grp_indices, 'scene_name'] = name

print(f"{len(groups)} groupes temporels créés")
for gid, grp_indices in enumerate(groups):
    name = rows.loc[grp_indices[0], 'scene_name']
    t0   = rows.loc[grp_indices[0],  'exif_date']
    t1   = rows.loc[grp_indices[-1], 'exif_date']
    print(f"  [{gid}] {name:40s} {len(grp_indices):3d} photos  ({t0:%H:%M} → {t1:%H:%M})")

25 groupes temporels créés
  [0] Matin · 22 juin 2022                       1 photos  (10:36 → 10:36)
  [1] Soirée · 21 jan 2023                       2 photos  (21:23 → 21:24)
  [2] Après-midi · 14 fév 2023                   1 photos  (17:56 → 17:56)
  [3] Nuit · 27 mai 2023                         1 photos  (00:16 → 00:16)
  [4] Soirée · 17 juil 2023                      1 photos  (18:08 → 18:08)
  [5] Matin · 16 août 2023                       1 photos  (06:30 → 06:30)
  [6] Nuit · 14 sep 2023                         1 photos  (05:59 → 05:59)
  [7] Après-midi · 16 déc 2023                   1 photos  (17:09 → 17:09)
  [8] Après-midi · 1 fév 2024                    5 photos  (12:31 → 17:31)
  [9] Après-midi · 7 mar 2024                    1 photos  (13:13 → 13:13)
  [10] Soirée · 4 avr 2024                        1 photos  (21:10 → 21:10)
  [11] Matin · 28 avr 2024                        1 photos  (08:01 → 08:01)
  [12] Nuit · 29 avr 2024                         2 photos  (01:05 → 01

In [6]:
# ── 5. RATTACHEMENT DES PHOTOS SANS DATE ─────────────────────────────────────
# Stratégie (par ordre de priorité) :
#   1. Date extraite du nom de fichier (patterns smartphones)
#   2. Similarité cosine avec seuil strict (≥ 0.60)
#   3. Groupe "Moment sans date" si aucune méthode ne conclut
#
# ⚠️  Ancienne version : pas de seuil → photos de dates radicalement différentes
#     pouvaient atterrir dans le même cluster (bug signalé).

import re

_DATE_PATTERNS = [
    (r'(\d{8})_(\d{6})',       'compact8_6'),   # IMG_20240927_205638
    (r'(\d{4})-(\d{2})-(\d{2})', 'iso'),        # 2024-09-27
    (r'(?<!\d)(\d{8})(?!\d)',  'compact8'),      # 20240927 seul
]

def _date_from_filename(filename: str):
    """Essaie d'extraire une date depuis le nom de fichier."""
    base = os.path.basename(filename)
    for pat, kind in _DATE_PATTERNS:
        m = re.search(pat, base)
        if not m:
            continue
        try:
            g = m.groups()
            if kind == 'compact8_6':
                raw = g[0]
                y, mo, d = int(raw[:4]), int(raw[4:6]), int(raw[6:8])
            elif kind == 'iso':
                y, mo, d = int(g[0]), int(g[1]), int(g[2])
            else:  # compact8
                raw = g[0]
                y, mo, d = int(raw[:4]), int(raw[4:6]), int(raw[6:8])
            if 2000 <= y <= 2030 and 1 <= mo <= 12 and 1 <= d <= 31:
                return pd.Timestamp(y, mo, d, 12, 0, 0)
        except Exception:
            continue
    return None


if len(undated) == 0:
    print("Toutes les photos ont une date EXIF — rien à rattacher")
elif len(groups) == 0:
    rows.loc[undated.index, 'scene_id']   = 0
    rows.loc[undated.index, 'scene_name'] = 'Moment sans date'
    print("Aucun groupe temporel — toutes les photos dans 'Moment sans date'")
else:
    # ── Centroids visuels par groupe ─────────────────────────────────────────
    group_centroids = {}
    for gid, grp_indices in enumerate(groups):
        embs = np.stack([
            np.array(rows.loc[i, 'embedding'], dtype=np.float32)
            for i in grp_indices
        ])
        c = embs.mean(axis=0)
        group_centroids[gid] = c / (np.linalg.norm(c) + 1e-8)

    centroids_matrix = np.stack(
        [group_centroids[i] for i in range(len(groups))]
    )

    # ── Midpoints temporels de chaque groupe ────────────────────────────────
    group_mids = []
    for gid, grp_indices in enumerate(groups):
        t0 = rows.loc[grp_indices[0],  'exif_date']
        t1 = rows.loc[grp_indices[-1], 'exif_date']
        group_mids.append(t0 + (t1 - t0) / 2)

    VISUAL_THRESHOLD = 0.60   # similarité cosine minimale pour l'assignation visuelle

    by_filename = 0
    by_visual   = 0
    to_undated  = 0

    for idx in undated.index:
        path_val = str(rows.loc[idx, 'path'] or '')
        fn_val   = str(rows.loc[idx, 'filename'] or path_val)
        assigned = False

        # ── 1. Date dans le nom de fichier ───────────────────────────────────
        fn_date = _date_from_filename(fn_val) or _date_from_filename(path_val)
        if fn_date is not None:
            deltas      = [abs((fn_date - mid).total_seconds()) for mid in group_mids]
            nearest_gid = int(np.argmin(deltas))
            rows.loc[idx, 'scene_id']   = nearest_gid
            rows.loc[idx, 'scene_name'] = rows.loc[groups[nearest_gid][0], 'scene_name']
            by_filename += 1
            assigned = True

        # ── 2. Similarité cosine avec seuil ─────────────────────────────────
        if not assigned:
            emb    = np.array(rows.loc[idx, 'embedding'], dtype=np.float32)
            emb_n  = emb / (np.linalg.norm(emb) + 1e-8)
            sims   = centroids_matrix @ emb_n
            best   = int(np.argmax(sims))
            best_sim = float(sims[best])

            if best_sim >= VISUAL_THRESHOLD:
                rows.loc[idx, 'scene_id']   = best
                rows.loc[idx, 'scene_name'] = rows.loc[groups[best][0], 'scene_name']
                by_visual += 1
                assigned = True

        # ── 3. Groupe "sans date" (similarité insuffisante) ──────────────────
        if not assigned:
            rows.loc[idx, 'scene_id']   = -2   # marqueur temporaire
            rows.loc[idx, 'scene_name'] = 'Moment sans date'
            to_undated += 1

    # Attribuer un vrai scene_id au groupe "sans date" s'il existe
    if (rows['scene_id'] == -2).any():
        undated_sid = int(rows[rows['scene_id'] >= 0]['scene_id'].max()) + 1
        rows.loc[rows['scene_id'] == -2, 'scene_id'] = undated_sid
        print(f"Nouveau groupe [scene_id={undated_sid}] créé pour les photos sans date")

    print(f"\n{len(undated)} photos sans date rattachées :")
    print(f"  {by_filename:2d}  par date dans le nom de fichier")
    print(f"  {by_visual:2d}  par similarité visuelle (seuil {VISUAL_THRESHOLD})")
    print(f"  {to_undated:2d}  → groupe 'Moment sans date' (non classifiables)")

Nouveau groupe [scene_id=25] créé pour les photos sans date

15 photos sans date rattachées :
   0  par date dans le nom de fichier
  11  par similarité visuelle (seuil 0.6)
   4  → groupe 'Moment sans date' (non classifiables)


In [7]:
# ── 6. RÉCAP FINAL ────────────────────────────────────────────────────────────
rows['scene_id'] = rows['scene_id'].astype(int)
rows['is_noise'] = False

print("\n── Groupes finaux ────────────────────────────────")
for sid in sorted(rows['scene_id'].unique()):
    grp  = rows[rows['scene_id'] == sid]
    name = grp['scene_name'].iloc[0]
    n    = len(grp)
    n_d  = grp['exif_date'].notna().sum()
    print(f"  [{sid}] {name:40s} {n:3d} photos  ({n_d} datées)")


── Groupes finaux ────────────────────────────────
  [0] Matin · 22 juin 2022                       1 photos  (1 datées)
  [1] Soirée · 21 jan 2023                       2 photos  (2 datées)
  [2] Après-midi · 14 fév 2023                   1 photos  (1 datées)
  [3] Nuit · 27 mai 2023                         1 photos  (1 datées)
  [4] Soirée · 17 juil 2023                      1 photos  (1 datées)
  [5] Matin · 16 août 2023                       1 photos  (1 datées)
  [6] Nuit · 14 sep 2023                         1 photos  (1 datées)
  [7] Après-midi · 16 déc 2023                   1 photos  (1 datées)
  [8] Après-midi · 1 fév 2024                    5 photos  (5 datées)
  [9] Après-midi · 7 mar 2024                    1 photos  (1 datées)
  [10] Soirée · 4 avr 2024                        1 photos  (1 datées)
  [11] Matin · 28 avr 2024                        1 photos  (1 datées)
  [12] Nuit · 29 avr 2024                         3 photos  (2 datées)
  [13] Nuit · 3 juil 2024          

### Écriture warehouse — scenes

**Merge key**: `photo_id` (identifiant unique par photo)  
**Stratégie**: Delta MERGE INTO — met a jour la scene_id/scene_name si une photo est re-attribuee, insere les nouvelles photos.  
**Idempotent**: oui — relancer le notebook ne crée pas de doublons de photos.

In [8]:
# -- 7. ECRITURE SCENES (Delta MERGE) --
from delta.tables import DeltaTable

df_scenes = spark.createDataFrame(
    rows[['photo_id', 'path', 'filename', 'exif_date',
          'lat', 'lon', 'caption', 'scene_id', 'scene_name', 'is_noise']]
    .astype({'scene_id': int})
    .where(rows['path'].notna())
)

# MERGE INTO si la table existe, sinon creation initiale
if DeltaTable.isDeltaTable(spark, SCENES_DIR):
    delta_tbl = DeltaTable.forPath(spark, SCENES_DIR)
    (
        delta_tbl.alias("target")
        .merge(df_scenes.alias("source"), "target.photo_id = source.photo_id")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
else:
    df_scenes.write.format('delta').mode('overwrite').save(SCENES_DIR)

print(f"Table scenes --> {SCENES_DIR}")

Table scenes --> /opt/spark/data/warehouse/memory_album/scenes


### Écriture warehouse — scene_centroids

**Merge key**: `scene_id` (identifiant unique par scene/cluster)  
**Stratégie**: Delta MERGE INTO — met a jour le centroid si les photos d une scene changent, insere les nouvelles scenes.  
**Idempotent**: oui — relancer le notebook ne crée pas de doublons de scenes.

In [ ]:
# -- 8. ECRITURE CENTROIDS (Delta MERGE) --
from delta.tables import DeltaTable

centroids_rows = []

for sid in sorted(rows['scene_id'].unique()):
    mask   = rows['scene_id'] == sid
    subset = rows[mask]

    embs     = np.stack(subset['embedding'].apply(
        lambda e: np.array(e, dtype=np.float32)
    ).values)
    centroid = embs.mean(axis=0)

    norms   = np.linalg.norm(embs - centroid, axis=1)
    rep_idx = subset.index[np.argmin(norms)]

    dates  = subset['exif_date'].dropna()
    ts_min = dates.min() if len(dates) else None
    ts_max = dates.max() if len(dates) else None

    centroids_rows.append({
        'scene_id'               : int(sid),
        'scene_name'             : subset['scene_name'].iloc[0],
        'photo_count'            : int(mask.sum()),
        'centroid_embedding'     : centroid.tolist(),
        'representative_caption' : rows.loc[rep_idx, 'caption'],
        'representative_photo'   : rows.loc[rep_idx, 'path'],
        'timestamp_start'        : ts_min,
        'timestamp_end'          : ts_max,
        'photo_ids'              : subset['photo_id'].tolist(),
    })

df_centroids = spark.createDataFrame(pd.DataFrame(centroids_rows))

# MERGE INTO si la table existe, sinon creation initiale
if DeltaTable.isDeltaTable(spark, CENTROIDS_DIR):
    delta_tbl = DeltaTable.forPath(spark, CENTROIDS_DIR)
    (
        delta_tbl.alias("target")
        .merge(df_centroids.alias("source"), "target.scene_id = source.scene_id")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
else:
    df_centroids.write.format('delta').mode('overwrite').save(CENTROIDS_DIR)

print(f"Table centroids --> {CENTROIDS_DIR}")

print("Recap groupes ─────────────────────────────────")
df_centroids.select(
    'scene_id', 'scene_name', 'photo_count',
    'timestamp_start', 'timestamp_end'
).orderBy('timestamp_start').show(truncate=60)

In [20]:
spark.stop()